# ENGS108 H5: 4,000-row inference simulation

Generated by `submission/h5/sim4k/build_sim.py`. Runs the code cells of `rs2027-engs108-submit-h5.ipynb` unchanged (real model, no stubs) on a synthetic 4,000-row test.csv; only `INPUT_ROOT` points at the synthetic mount. Not a competition submission.

In [ ]:
import glob, subprocess, sys
wheels = sorted(glob.glob("/kaggle/input/**/bitsandbytes-*.whl", recursive=True))
assert wheels, "attach the engs108-offline-wheels dataset"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheels[-1]], check=True)


In [ ]:
import glob, os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

N_ROWS, N_MAXLEN_BLOCK, N_MAXLEN_SPREAD, N_EMPTY = 4000, 64, 136, 8
train_csv = sorted(glob.glob("/kaggle/input/competitions/*/train.csv") + glob.glob("/kaggle/input/*/train.csv"))[0]
real_sample = os.path.join(os.path.dirname(train_csv), "sample_submission.parquet")
train = pd.read_csv(train_csv)
rng = np.random.default_rng(0)
pool = train["text"].astype(str).to_numpy()
# max-length passages: 6 concatenated train texts (about 4,500 chars, far above 512 tokens)
maxlen = ["\n\n".join(rng.choice(pool, 6, replace=False)) for _ in range(N_MAXLEN_BLOCK + N_MAXLEN_SPREAD)]
regular = list(rng.choice(pool, N_ROWS - len(maxlen) - N_EMPTY))
spread = regular + maxlen[N_MAXLEN_BLOCK:] + [""] * (N_EMPTY // 2) + [None] * (N_EMPTY // 2)
spread = [spread[i] for i in rng.permutation(len(spread))]
mid = len(spread) // 2
texts = spread[:mid] + maxlen[:N_MAXLEN_BLOCK] + spread[mid:]  # 8 contiguous batches of 8 max-length texts
ids = [f"{i:012x}" for i in rng.choice(16 ** 10, N_ROWS, replace=False)]
SIM_DIR = "/kaggle/working/sim_input/competitions/name-that-llm"
os.makedirs(SIM_DIR, exist_ok=True)
pd.DataFrame({"id": ids, "text": texts}).to_csv(f"{SIM_DIR}/test.csv", index=False)
schema = pq.read_schema(real_sample)
pq.write_table(pa.table({"id": ids, "model": ["gpt-5.5"] * N_ROWS}).cast(schema), f"{SIM_DIR}/sample_submission.parquet")
SIM_INFO = {"rows": N_ROWS, "maxlen_rows": len(maxlen), "maxlen_contiguous_block_rows": N_MAXLEN_BLOCK,
            "maxlen_block_start_row": mid, "empty_or_null_rows": N_EMPTY, "source": train_csv,
            "text_chars": pd.Series([len(t or "") for t in texts]).describe().to_dict()}
print(SIM_INFO)


In [ ]:
import json, time, traceback

CELLS = json.loads(r"""["import os\n\nos.environ[\"CUDA_VISIBLE_DEVICES\"] = \"0\"  # one T4 (as the Arm H kernel)\n\nimport glob, hashlib, json, sys, time\nimport importlib.metadata as metadata\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\nimport torch\n\nif not torch.cuda.is_available():\n    print(\"!\" * 100 + \"\\nWARNING: torch.cuda.is_available() is False: this session has NO GPU. The H5 notebook needs a T4 \"\n          \"(4-bit Qwen3-8B); a CPU run would OOM or fail later. Enable Accelerator = GPU T4 for this notebook.\\n\"\n          + \"!\" * 100, flush=True)\n\nRUN_START = time.time()\nCONFIG = json.loads(r\"\"\"{\n \"submission\": \"H5: equal-weight log-linear blend of probe_armh + armh_head + stylo_exp03 (exp03 pre-registered report H5)\",\n \"inference_only\": true,\n \"run_dev_check\": false,\n \"seed\": 42,\n \"class_names\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\",\n                 \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\",\n                 \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\",\n                 \"kimi-k3\", \"qwen3.8-max\"],\n \"model\": {\n  \"base_path\": \"/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1\",\n  \"class\": \"Qwen3ForSequenceClassification via AutoModelForSequenceClassification(num_labels=18, id2label = alphabetical)\",\n  \"quantization\": {\"load_in_4bit\": true, \"bnb_4bit_quant_type\": \"nf4\", \"bnb_4bit_compute_dtype\": \"float16\",\n                   \"bnb_4bit_use_double_quant\": true},\n  \"adapter\": \"engs108-exp03-armh-artifacts/adapter (Arm H epoch 6 = checkpoint-858, modules_to_save=[score])\",\n  \"max_length\": 512,\n  \"batch_size\": 8,\n  \"autocast\": \"float16\"\n },\n \"forward\": \"one shared pass per batch: exp03 armh_core.pooled_forward (hook on the score input, last non-pad token) returns the head logits and the pooled hidden state of the same pass; batches of 8 in file order, which is the order the Arm H kernel used for pooled_hidden_armh.npy (the probe's training/dev features)\",\n \"blend\": {\"id\": \"H5\", \"components\": [\"probe_armh\", \"armh_head\", \"stylo_exp03\"], \"weights\": [1.0, 1.0, 1.0],\n           \"method\": \"exp03/stylo/protocol.py logprob_blend: softmax(sum_k log clip(p_k, 1e-12, 1))\",\n           \"not\": \"H6 dev-LOPO weights are NOT used\"},\n \"dev_check\": {\n  \"rows\": 414,\n  \"prompts\": 23,\n  \"expected_balanced_accuracy\": 0.4323671497584541,\n  \"tolerance\": 0.001,\n  \"require_argmax_agreement_vs_blend_H5_dev\": 1.0,\n  \"max_abs_proba_diff_vs_shared_pass_reference\": 0.01\n },\n \"sha256\": {\n  \"train.csv\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\",\n  \"adapter/adapter_model.safetensors\": \"dae9601c1f5cfba01da3c1c189c893a7e011ed274e26c50a775fe51a42084354\",\n  \"adapter/adapter_config.json\": \"b774b665076a1ba2cb0aa34e299815cbf058cf17ac64ab56139932617e0a8a3a\",\n  \"adapter/label_map.json\": \"820b6c166aa933d1c92902dbcf4f380b188b0de1a4d6e9e228a7e30ac307aff8\",\n  \"probe_armh.npz\": \"12dc23b8218447a9148407583b1c5fe015e23023a04e387ad66257c1b3604093\",\n  \"stylo_exp03_model.joblib\": \"4377983503ab5b4afb6675f273ca21f1699c1b9f46bad7c311212e25d1ee87df\",\n  \"blend_H5_dev.csv\": \"2dcc47082c9089897fd220ebadc52ec2960ae2d639a9e1c0ea78c2ef224fba73\",\n  \"h5_dev_reference.csv\": \"bb1a2640f5ee2df0b5b28e33b26c095abc72395e7fb549d4b5a563cef1bf7f27\"\n },\n \"kaggle\": {\n  \"kernel\": \"ritisingh123/rs2027-engs108-submit-h5\",\n  \"competition\": \"name-that-llm\",\n  \"datasets\": [\"ritisingh123/engs108-offline-wheels\", \"ritisingh123/engs108-exp03-armh-artifacts\",\n               \"ritisingh123/engs108-submission-h5-artifacts\"],\n  \"artifacts_dataset\": \"ritisingh123/engs108-submission-h5-artifacts\",\n  \"model_sources\": [\"qwen-lm/qwen-3/Transformers/8b-base/1\"],\n  \"docker_image\": \"gcr.io/kaggle-private-byod/python@sha256:de153e6d303881930c51286216393d56f56b26458d0661350305a1e8144eea01\",\n  \"machine_shape\": \"NvidiaTeslaT4\",\n  \"internet\": false,\n  \"gpu\": true,\n  \"leaderboard_submission\": false\n }\n}\n\"\"\")\nMC, DC, SHA = CONFIG[\"model\"], CONFIG[\"dev_check\"], CONFIG[\"sha256\"]\nSEED, MAX_LENGTH, BATCH, N_CLASSES = CONFIG[\"seed\"], MC[\"max_length\"], MC[\"batch_size\"], len(CONFIG[\"class_names\"])\n# Optional 414-row dev reproduction (diagnostic). Off for the official run: then train.csv is never hashed or read.\nRUN_DEV_CHECK = CONFIG.get(\"run_dev_check\", False)\nINPUT_ROOT = \"/kaggle/working/sim_input\"\nMODEL_PATH = MC[\"base_path\"]\nOUT = \"/kaggle/working\"\n\nversions = {p: metadata.version(p) for p in\n            [\"torch\", \"transformers\", \"peft\", \"accelerate\", \"bitsandbytes\", \"scikit-learn\", \"numpy\", \"pandas\", \"scipy\"]}\nprint(versions)\nprint(\"CUDA:\", torch.cuda.is_available(), \"| visible GPUs:\", torch.cuda.device_count(),\n      torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"\")\nassert torch.cuda.is_available(), \"GPU required\"\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef find_one(pattern):\n    hits = sorted(glob.glob(pattern, recursive=True))\n    assert len(hits) == 1, (pattern, hits)\n    return hits[0]\n\n\ndef find_data_dir(root, slug=\"name-that-llm\"):\n    \"\"\"Competition data dir: the known mounts, then any competitions/* mount, then any top-level mount, holding BOTH\n    test.csv and sample_submission.parquet with the expected columns. Dataset mounts (datasets/<owner>/<slug>) are never\n    candidates, so an attached dataset with similarly named files cannot be picked.\"\"\"\n    candidates = [f\"{root}/competitions/{slug}\", f\"{root}/{slug}\"]\n    candidates += sorted(glob.glob(f\"{root}/competitions/*/\")) + sorted(glob.glob(f\"{root}/*/\"))\n    tried = []\n    for d in dict.fromkeys(os.path.normpath(c) for c in candidates):\n        missing = [f for f in (\"test.csv\", \"sample_submission.parquet\") if not os.path.isfile(f\"{d}/{f}\")]\n        if missing:\n            tried.append(f\"{d}: missing {missing}\")\n            continue\n        try:\n            cols = set(pd.read_csv(f\"{d}/test.csv\").columns)\n            names = pq.read_schema(f\"{d}/sample_submission.parquet\").names\n        except Exception as e:\n            tried.append(f\"{d}: unreadable ({type(e).__name__}: {e})\")\n            continue\n        if not {\"id\", \"text\"} <= cols or names != [\"id\", \"model\"]:\n            tried.append(f\"{d}: test.csv columns {sorted(cols)}, sample_submission.parquet names {names}\")\n            continue\n        return d, tried\n    raise AssertionError(\"no competition data dir with test.csv (id, text) + sample_submission.parquet (id, model) \"\n                         f\"under {root}; candidates tried:\\n  \" + \"\\n  \".join(tried))\n\n\nDATA_DIR, data_dir_rejected = find_data_dir(INPUT_ROOT)\nprint(\"DATA_DIR:\", DATA_DIR, \"| rejected candidates before it:\", data_dir_rejected)\nADAPTER_DIR = os.path.dirname(find_one(\"/kaggle/input/**/engs108-exp03-armh-artifacts/adapter/adapter_config.json\"))\nH5_DIR = os.path.dirname(find_one(\"/kaggle/input/**/engs108-submission-h5-artifacts/probe_armh.npz\"))\nprint(\"adapter:\", ADAPTER_DIR, sorted(os.listdir(ADAPTER_DIR)))\nprint(\"h5 artifacts:\", H5_DIR, sorted(os.listdir(H5_DIR)))\n", "FEATURES_SRC = r'''\"\"\"exp03 stylometric feature groups.\n\nGroups (names used in configs and ablations):\n  char    char_wb 3-5 TF-IDF (exp02)\n  word    word 1-2 TF-IDF (exp02)\n  style   27 exp02 style features (copied verbatim from exp02; test_features checks equality)\n  fw      relative frequencies of a fixed English function-word list (dense)\n  fwng    function-word n-grams: content words masked by their shape, TF-IDF 1-3 grams\n  typo    typography / formatting rates (dense)\n  sent    sentence / paragraph structure statistics and word-length histogram (dense)\n  shape   token-shape n-grams (Xx, x, X, d, punctuation literal, newline), TF-IDF 1-3 grams\nDense groups are standardised; TF-IDF groups are used as is. Every learned step is a sklearn\ntransformer, so it is refit inside each CV fold.\n\"\"\"\nimport re\nimport string\n\nimport numpy as np\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import FunctionTransformer, StandardScaler\n\n# ---------------------------------------------------------------- exp02 style features (verbatim)\nPUNCT = set(string.punctuation) | set(\"\u2014\u2013\u2018\u2019\u201c\u201d\u2026\u00ab\u00bb\u00b7\u2022\")\nSTYLE_NAMES = [\"log_chars\", \"log_words\", \"mean_word_len\", \"n_sentences\", \"mean_sentence_words\", \"type_token_ratio\",\n               \"punct_rate\", \"upper_rate\", \"digit_rate\", \"whitespace_rate\", \"newline_rate\",\n               \"comma_rate\", \"semicolon_rate\", \"colon_rate\", \"em_dash_rate\", \"en_dash_rate\", \"hyphen_rate\",\n               \"curly_apos_rate\", \"straight_apos_rate\", \"curly_quote_rate\", \"straight_quote_rate\",\n               \"paren_rate\", \"exclaim_rate\", \"question_rate\", \"ellipsis_rate\", \"asterisk_rate\", \"hash_rate\"]\n\n\ndef style_features(texts):\n    rows = []\n    for t in texts:\n        n = max(len(t), 1)\n        words = re.findall(r\"\\w+\", t)\n        sents = [s for s in re.split(r\"[.!?]+\\s\", t) if s.strip()]\n        rows.append([\n            np.log1p(len(t)), np.log1p(len(words)), np.mean([len(w) for w in words]) if words else 0.0,\n            len(sents), len(words) / max(len(sents), 1), len({w.lower() for w in words}) / max(len(words), 1),\n            sum(c in PUNCT for c in t) / n, sum(c.isupper() for c in t) / n, sum(c.isdigit() for c in t) / n,\n            sum(c.isspace() for c in t) / n, t.count(\"\\n\") / n,\n            t.count(\",\") / n, t.count(\";\") / n, t.count(\":\") / n, t.count(\"\u2014\") / n, t.count(\"\u2013\") / n, t.count(\"-\") / n,\n            t.count(\"\u2019\") / n, t.count(\"'\") / n, (t.count(\"\u201c\") + t.count(\"\u201d\")) / n, t.count('\"') / n,\n            (t.count(\"(\") + t.count(\")\")) / n, t.count(\"!\") / n, t.count(\"?\") / n,\n            (t.count(\"\u2026\") + t.count(\"...\")) / n, t.count(\"*\") / n, t.count(\"#\") / n,\n        ])\n    return np.asarray(rows, dtype=np.float64)\n\n\n# ---------------------------------------------------------------- function words\n# Closed-class English words: determiners, pronouns, prepositions, conjunctions, auxiliaries,\n# common adverbs/particles and discourse markers. Fixed a priori (not learned from the data).\nFUNCTION_WORDS = sorted(set(\"\"\"\na an the this that these those some any each every either neither no another such what which whose\ni me my mine myself we us our ours ourselves you your yours yourself yourselves he him his himself she her hers\nherself it its itself they them their theirs themselves one ones oneself who whom whoever whatever whichever\nabout above across after against along amid among around as at before behind below beneath beside besides\nbetween beyond but by despite down during except for from in inside into like near of off on onto out outside\nover past per since than through throughout till to toward towards under underneath unlike until up upon via\nwith within without\nand or nor so yet both whether because although though while whereas if unless once whenever wherever\nhowever therefore thus hence moreover furthermore nevertheless nonetheless meanwhile instead otherwise indeed\nalso still even just only already again ever never always often sometimes rather quite very too much more most\nless least enough almost perhaps maybe merely simply really actually particularly especially\nbe am is are was were been being have has had having do does did doing will would shall should can could\nmay might must ought need dare\nnot here there where when why how then now well all both few many several most other others same own\n\"\"\".split()))\n_WORD_RE = re.compile(r\"[A-Za-z]+(?:['\u2019][A-Za-z]+)?\")\n_FW_INDEX = {w: i for i, w in enumerate(FUNCTION_WORDS)}\n\n\ndef _words(t):\n    return _WORD_RE.findall(t)\n\n\ndef function_word_features(texts):\n    \"\"\"Relative frequency (per word) of each function word; contractions split on the apostrophe.\"\"\"\n    out = np.zeros((len(texts), len(FUNCTION_WORDS)))\n    for r, t in enumerate(texts):\n        ws = [w.lower().replace(\"\u2019\", \"'\").split(\"'\")[0] for w in _words(t)]\n        for w in ws:\n            j = _FW_INDEX.get(w)\n            if j is not None:\n                out[r, j] += 1\n        out[r] /= max(len(ws), 1)\n    return out\n\n\ndef _word_shape(w):\n    if w.isdigit():\n        return \"d\"\n    if w.isupper() and len(w) > 1:\n        return \"X\"\n    if w[0].isupper():\n        return \"Xx\"\n    return \"x\"\n\n\n_TOKEN_RE = re.compile(r\"\\n|[A-Za-z]+(?:['\u2019][A-Za-z]+)?|\\d+(?:[.,]\\d+)*|[^\\w\\s]\")\n\n\ndef _tokens(t):\n    return _TOKEN_RE.findall(t)\n\n\ndef fw_masked_docs(texts):\n    \"\"\"Keep function words (lower-cased) and punctuation; replace other tokens by their shape.\"\"\"\n    docs = []\n    for t in texts:\n        out = []\n        for tok in _tokens(t):\n            if tok == \"\\n\":\n                out.append(\"NL\")\n            elif tok[0].isalpha():\n                base = tok.lower().replace(\"\u2019\", \"'\").split(\"'\")[0]\n                out.append(base if base in _FW_INDEX else \"<\" + _word_shape(tok) + \">\")\n            elif tok[0].isdigit():\n                out.append(\"<d>\")\n            else:\n                out.append(tok)\n        docs.append(\" \".join(out))\n    return docs\n\n\ndef shape_docs(texts):\n    \"\"\"Token-shape sequence: word shape + length bucket (s<=3, m<=6, l), digits 'd', punctuation literal.\"\"\"\n    docs = []\n    for t in texts:\n        out = []\n        for tok in _tokens(t):\n            if tok == \"\\n\":\n                out.append(\"NL\")\n            elif tok[0].isalpha():\n                L = len(tok)\n                out.append(_word_shape(tok) + (\"s\" if L <= 3 else \"m\" if L <= 6 else \"l\")\n                           + (\"'\" if (\"'\" in tok or \"\u2019\" in tok) else \"\"))\n            elif tok[0].isdigit():\n                out.append(\"d\")\n            else:\n                out.append(tok)\n        docs.append(\" \".join(out))\n    return docs\n\n\n# ---------------------------------------------------------------- typography\nTYPO_NAMES = [\"non_ascii_rate\", \"curly_share_of_quotes\", \"curly_share_of_apos\", \"spaced_em_dash_rate\",\n              \"unspaced_em_dash_rate\", \"spaced_hyphen_dash_rate\", \"double_hyphen_rate\", \"double_space_rate\",\n              \"space_before_punct_rate\", \"bold_md_rate\", \"italic_md_rate\", \"bullet_line_frac\", \"numbered_line_frac\",\n              \"heading_line_frac\", \"blank_line_rate\", \"contraction_rate\", \"hyphenated_word_rate\", \"title_case_word_rate\",\n              \"all_caps_word_rate\", \"oxford_comma_rate\", \"eg_ie_rate\", \"percent_rate\", \"ampersand_rate\", \"slash_rate\",\n              \"colon_then_capital_rate\", \"semicolon_per_sentence\", \"ends_with_period\", \"ends_with_other\",\n              \"starts_with_quote\", \"trailing_space_line_frac\", \"number_token_rate\", \"ellipsis_char_share\"]\n\n\ndef typography_features(texts):\n    rows = []\n    for t in texts:\n        n = max(len(t), 1)\n        words = _words(t)\n        nw = max(len(words), 1)\n        lines = t.split(\"\\n\")\n        nl = max(len(lines), 1)\n        quotes_curly = t.count(\"\u201c\") + t.count(\"\u201d\")\n        quotes_all = quotes_curly + t.count('\"')\n        apos_curly = t.count(\"\u2019\")\n        apos_all = apos_curly + t.count(\"'\")\n        n_sent = max(len([s for s in re.split(r\"[.!?]+\\s\", t) if s.strip()]), 1)\n        ell_char, ell_dots = t.count(\"\u2026\"), t.count(\"...\")\n        s = t.rstrip()\n        rows.append([\n            sum(ord(c) > 127 for c in t) / n,\n            quotes_curly / quotes_all if quotes_all else 0.5,\n            apos_curly / apos_all if apos_all else 0.5,\n            len(re.findall(r\" \u2014 \", t)) / n, len(re.findall(r\"\\S\u2014\\S\", t)) / n, len(re.findall(r\" - \", t)) / n,\n            t.count(\"--\") / n, t.count(\"  \") / n, len(re.findall(r\" [,.;:!?]\", t)) / n,\n            len(re.findall(r\"\\*\\*[^*]+\\*\\*\", t)) / n, len(re.findall(r\"(?<![*\\w])[*_][^*_\\n]+[*_](?![*\\w])\", t)) / n,\n            sum(bool(re.match(r\"\\s*[-*\u2022]\\s\", l)) for l in lines) / nl,\n            sum(bool(re.match(r\"\\s*\\d+[.)]\\s\", l)) for l in lines) / nl,\n            sum(bool(re.match(r\"\\s*#+\\s\", l)) for l in lines) / nl,\n            t.count(\"\\n\\n\") / n,\n            sum((\"'\" in w or \"\u2019\" in w) for w in words) / nw,\n            len(re.findall(r\"\\b[A-Za-z]+-[A-Za-z]+\\b\", t)) / nw,\n            sum(w[0].isupper() and w[1:].islower() for w in words if len(w) > 1) / nw,\n            sum(w.isupper() and len(w) > 1 for w in words) / nw,\n            len(re.findall(r\",\\s+(?:and|or)\\b\", t)) / nw,\n            len(re.findall(r\"\\b(?:e\\.g\\.|i\\.e\\.|etc\\.)\", t)) / nw,\n            t.count(\"%\") / n, t.count(\"&\") / n, t.count(\"/\") / n,\n            len(re.findall(r\":\\s+[A-Z]\", t)) / n, t.count(\";\") / n_sent,\n            float(s.endswith(\".\")), float(bool(s) and s[-1] not in \".!?\\\"\u201d'\u2019)\"),\n            float(t.lstrip()[:1] in \"\\\"\u201c'\u2018\"),\n            sum(l != l.rstrip(\" \\t\") for l in lines) / nl,\n            len(re.findall(r\"\\b\\d\", t)) / nw,\n            ell_char / (ell_char + ell_dots) if (ell_char + ell_dots) else 0.5,\n        ])\n    return np.asarray(rows, dtype=np.float64)\n\n\n# ---------------------------------------------------------------- sentence / paragraph structure\n_SENT_SPLIT = re.compile(r\"(?<=[.!?\u2026])[\\\"\u201d\u2019)]*\\s+(?=[\\\"\u201c\u2018(]*[A-Z0-9])\")\nSENT_NAMES = ([\"sent_len_mean\", \"sent_len_std\", \"sent_len_cv\", \"sent_len_median\", \"sent_len_p10\", \"sent_len_p90\",\n               \"sent_len_min\", \"sent_len_max\", \"n_sentences_log\", \"frac_short_sent\", \"frac_long_sent\",\n               \"first_sent_len\", \"last_sent_len\", \"commas_per_sent\", \"frac_question_sent\", \"frac_exclaim_sent\",\n               \"frac_conj_start\", \"frac_the_start\", \"frac_pronoun_start\", \"frac_adverb_ly_start\",\n               \"frac_sent_with_colon\", \"frac_sent_with_dash\", \"frac_sent_with_paren\",\n               \"n_paragraphs_log\", \"sent_per_para_mean\", \"words_per_para_mean\", \"words_per_para_std\",\n               \"word_len_std\", \"long_word_rate\", \"hapax_rate\", \"mean_word_freq_rank_proxy\"]\n              + [f\"wordlen_{k}\" for k in range(1, 16)])\n_CONJ = {\"and\", \"but\", \"or\", \"so\", \"yet\", \"nor\"}\n_PRON = {\"i\", \"we\", \"you\", \"he\", \"she\", \"it\", \"they\", \"this\", \"that\", \"these\", \"those\"}\n\n\ndef sentence_features(texts):\n    rows = []\n    for t in texts:\n        sents = [s for s in _SENT_SPLIT.split(t.strip()) if s.strip()] or [t]\n        sw = [_words(s) for s in sents]\n        L = np.array([len(w) for w in sw], dtype=float)\n        first = [w[0].lower() if w else \"\" for w in sw]\n        paras = [p for p in re.split(r\"\\n\\s*\\n\", t.strip()) if p.strip()] or [t]\n        pw = np.array([len(_words(p)) for p in paras], dtype=float)\n        words = _words(t)\n        wl = np.array([len(w) for w in words], dtype=float) if words else np.zeros(1)\n        lower = [w.lower() for w in words]\n        counts = {}\n        for w in lower:\n            counts[w] = counts.get(w, 0) + 1\n        nw = max(len(words), 1)\n        ns = len(sents)\n        hist = np.bincount(np.clip(wl.astype(int), 1, 15), minlength=16)[1:16] / nw\n        rows.append([\n            L.mean(), L.std(), L.std() / max(L.mean(), 1e-9), np.median(L), np.percentile(L, 10), np.percentile(L, 90),\n            L.min(), L.max(), np.log1p(ns), (L < 8).mean(), (L > 30).mean(), L[0], L[-1],\n            sum(s.count(\",\") for s in sents) / ns, np.mean([s.rstrip().endswith(\"?\") for s in sents]),\n            np.mean([s.rstrip().endswith(\"!\") for s in sents]),\n            np.mean([f in _CONJ for f in first]), np.mean([f == \"the\" for f in first]),\n            np.mean([f in _PRON for f in first]), np.mean([f.endswith(\"ly\") and len(f) > 3 for f in first]),\n            np.mean([\":\" in s for s in sents]), np.mean([(\"\u2014\" in s or \" - \" in s or \"\u2013\" in s) for s in sents]),\n            np.mean([\"(\" in s for s in sents]),\n            np.log1p(len(paras)), ns / len(paras), pw.mean(), pw.std(),\n            wl.std(), (wl >= 7).mean(), sum(c == 1 for c in counts.values()) / nw,\n            np.mean([w in _FW_INDEX for w in lower]) if lower else 0.0,\n            *hist,\n        ])\n    return np.asarray(rows, dtype=np.float64)\n\n\n# ---------------------------------------------------------------- group registry\ndef _dense(fn, clip=None):\n    steps = [(\"f\", FunctionTransformer(fn)), (\"z\", StandardScaler())]\n    if clip is not None:\n        steps.append((\"clip\", FunctionTransformer(np.clip, kw_args={\"a_min\": -clip, \"a_max\": clip})))\n    return Pipeline(steps)\n\n\ndef fw_tokens(text):\n    \"\"\"Function-word tokens of a text (the analyzer of the `fw` group).\"\"\"\n    return [w for w in (x.lower().replace(\"\u2019\", \"'\").split(\"'\")[0] for x in _words(text)) if w in _FW_INDEX]\n\n\ndef _tfidf_docs(fn, **kw):\n    return Pipeline([(\"docs\", FunctionTransformer(fn)),\n                     (\"tfidf\", TfidfVectorizer(token_pattern=r\"\\S+\", lowercase=False, sublinear_tf=True, **kw))])\n\n\ndef make_group(name):\n    if name == \"char\":\n        return TfidfVectorizer(analyzer=\"char_wb\", ngram_range=(3, 5), min_df=2, sublinear_tf=True)\n    if name == \"word\":\n        return TfidfVectorizer(analyzer=\"word\", ngram_range=(1, 2), min_df=2, sublinear_tf=True)\n    if name == \"style\":\n        return _dense(style_features)\n    if name == \"fw\":\n        # sublinear TF-IDF over the fixed function-word vocabulary; standardised per-word rates gave rare\n        # function words huge z-scores on held-out rows (first ablation run: OOF log loss ~12)\n        return TfidfVectorizer(analyzer=fw_tokens, vocabulary=FUNCTION_WORDS, sublinear_tf=True)\n    if name == \"fwng\":\n        return _tfidf_docs(fw_masked_docs, ngram_range=(1, 3), min_df=3)\n    if name == \"typo\":\n        return _dense(typography_features, clip=DENSE_CLIP)\n    if name == \"sent\":\n        return _dense(sentence_features, clip=DENSE_CLIP)\n    if name == \"shape\":\n        return _tfidf_docs(shape_docs, ngram_range=(1, 3), min_df=3)\n    raise ValueError(name)\n\n\nDENSE_CLIP = 5.0\nGROUPS = [\"char\", \"word\", \"style\", \"fw\", \"fwng\", \"typo\", \"sent\", \"shape\"]\nEXP02_GROUPS = [\"char\", \"word\", \"style\"]\nNEW_GROUPS = [\"fw\", \"fwng\", \"typo\", \"sent\", \"shape\"]\n\n\ndef make_features(groups):\n    \"\"\"ColumnTransformer over a DataFrame with a 'text' column; block order = `groups` order.\"\"\"\n    return ColumnTransformer([(g, make_group(g), \"text\") for g in groups])\n'''\n\nfrom scipy.special import log_softmax, softmax\n\n# ---- exp03/stylo/features.py (written as a module: stylo_exp03_model.joblib pickles references to `features`) ----\nSRC_DIR = f\"{OUT}/h5_src\"\nos.makedirs(SRC_DIR, exist_ok=True)\nwith open(f\"{SRC_DIR}/features.py\", \"w\") as f:\n    f.write(FEATURES_SRC)\nsys.path.insert(0, SRC_DIR)\nimport features  # noqa: F401\n\n# ---- exp03/stylo/protocol.py (inlined: log_proba, logprob_blend) ----\nCLASS_NAMES = CONFIG[\"class_names\"]\nPROBA_FLOOR = 1e-12\n\n\ndef log_proba(p):\n    return np.log(np.clip(np.asarray(p, dtype=np.float64), PROBA_FLOOR, 1.0))\n\n\ndef logprob_blend(log_probs, weights=None):\n    \"\"\"Weighted log-linear pool: softmax(sum_k w_k log p_k). weights=None -> all 1 (equal blend, nothing fitted).\"\"\"\n    weights = np.ones(len(log_probs)) if weights is None else np.asarray(weights, dtype=np.float64)\n    z = sum(w * lp for w, lp in zip(weights, log_probs))\n    return np.exp(log_softmax(z, axis=1))\n\n# ---- exp03/probe/probe.py (inlined: ProbeModel) ----\nclass ProbeModel:\n    \"\"\"Numpy-only probe loaded from the exported .npz.\"\"\"\n\n    def __init__(self, path):\n        with np.load(path, allow_pickle=False) as d:\n            self.mean, self.scale = d[\"mean\"], d[\"scale\"]\n            self.coef, self.intercept = d[\"coef\"], d[\"intercept\"]\n            self.class_names = [str(c) for c in d[\"class_names\"]]\n            self.meta = json.loads(str(d[\"meta\"]))\n\n    def decision_function(self, H):\n        H = np.asarray(H, dtype=np.float64)\n        assert H.ndim == 2 and H.shape[1] == self.mean.shape[0], H.shape\n        return ((H - self.mean) / self.scale) @ self.coef.T + self.intercept\n\n    def predict_proba(self, H):\n        return softmax(self.decision_function(H), axis=1)\n\n# ---- exp03/qlora_armh/armh_core.py (inlined: pooled_forward) ----\ndef pooled_forward(model, score_module, tokenizer, texts, batch_size, max_length, n_classes, device=\"cuda:0\"):\n    \"\"\"exp02 pooled_forward: hidden state at the last non-pad token feeding `score`, plus the model logits.\"\"\"\n    captured = {}\n    handle = score_module.register_forward_hook(lambda mod, inp, out: captured.__setitem__(\"h\", inp[0].detach()))\n    hidden = np.zeros((len(texts), score_module.in_features if hasattr(score_module, \"in_features\")\n                       else score_module.original_module.in_features), dtype=np.float32)\n    logits = np.zeros((len(texts), n_classes), dtype=np.float32)\n    try:\n        with torch.no_grad():\n            for start in range(0, len(texts), batch_size):\n                batch = texts[start:start + batch_size]\n                enc = tokenizer(batch, truncation=True, max_length=max_length, padding=True, return_tensors=\"pt\").to(device)\n                with torch.autocast(device.split(\":\")[0], dtype=torch.float16, enabled=device.startswith(\"cuda\")):\n                    out = model(**enc).logits\n                ids = enc[\"input_ids\"]\n                non_pad = (ids != tokenizer.pad_token_id).int()\n                last = (torch.arange(ids.shape[-1], device=ids.device) * non_pad).argmax(-1)\n                h = captured[\"h\"][torch.arange(ids.shape[0], device=ids.device), last]\n                hidden[start:start + len(batch)] = h.float().cpu().numpy()\n                logits[start:start + len(batch)] = out.float().cpu().numpy()\n    finally:\n        handle.remove()\n    return hidden, logits\n\n# ---- submission/h5/h5_core.py (inlined) ----\n\"\"\"H5 submission logic that runs after the shared Qwen forward pass (CPU; inlined into the Kaggle notebook by\nbuild_notebook.py, tested offline in tests/).\n\nH5 = equal-weight log-linear pool of three components, exactly as in exp03/qlora_armh/analyze_armh.py:\n  probe_armh   C=0.03 StandardScaler+LR probe (probe_armh.npz) on the pooled score-input hidden state\n  armh_head    softmax of the epoch-6 Arm H classification head logits\n  stylo_exp03  char+word+style TF-IDF/LR pipeline (stylo_exp03_model.joblib)\nblend = protocol.logprob_blend([log p_probe, log p_head, log p_stylo]) (weights all 1, nothing fitted).\n\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\nfrom sklearn.metrics import balanced_accuracy_score\n\nH5_COMPONENTS = (\"probe_armh\", \"armh_head\", \"stylo_exp03\")\n\n\ndef component_probas(hidden, logits, texts, probe, stylo):\n    \"\"\"Per-component probabilities (n, 18) in CLASS_NAMES order from one shared forward pass.\n\n    The probe was fit and scored on pooled_hidden_armh.npy, which the Arm H kernel stored as float16, so the\n    hidden states take the same float32 -> float16 -> float64 path here.\"\"\"\n    H = np.asarray(hidden, dtype=np.float32).astype(np.float16).astype(np.float64)\n    return {\"probe_armh\": probe.predict_proba(H),\n            \"armh_head\": softmax(np.asarray(logits, dtype=np.float32).astype(np.float64), axis=1),\n            \"stylo_exp03\": stylo.predict_proba(pd.DataFrame({\"text\": list(texts)}))}\n\n\ndef h5_blend(comps):\n    \"\"\"Fixed equal-weight H5 blend (not the H6 dev-LOPO weights).\"\"\"\n    return logprob_blend([log_proba(comps[k]) for k in H5_COMPONENTS])\n\n\ndef check_class_order(label_map_names, model_config_names, probe_names, stylo_classes, train_models=None):\n    \"\"\"Every component must use the alphabetical 18-class order (class id c = c-th sorted model name).\n\n    train_models (train.csv \"model\") is only checked when given, i.e. when the optional dev check reads train.csv.\"\"\"\n    names = list(CLASS_NAMES)\n    assert len(names) == 18 and names == sorted(names)\n    assert list(label_map_names) == names, \"adapter label_map.json order differs\"\n    assert list(model_config_names) == names, \"model.config.id2label order differs\"\n    assert list(probe_names) == names, \"probe_armh.npz class_names order differs\"\n    assert [int(c) for c in stylo_classes] == list(range(18)), \"stylo classes_ are not label ids 0..17\"\n    if train_models is not None:\n        assert sorted(set(train_models)) == names, \"train.csv model names differ from CLASS_NAMES\"\n    return names\n\n\ndef dev_reproduction(y, blend, comps, ref_h5, ref_shared, ref_comps, expected_bal_acc, tol, max_dp_shared):\n    \"\"\"Compare the dev run against blend_H5_dev.csv (the reported H5) and the shared-pass reference.\n\n    Returns (report, failures). failures is empty when the run reproduces H5.\"\"\"\n    pred = blend.argmax(1)\n    bal = float(balanced_accuracy_score(y, pred))\n    rep = {\"balanced_accuracy\": bal, \"expected_balanced_accuracy\": expected_bal_acc, \"tolerance\": tol,\n           \"argmax_agreement_vs_blend_H5_dev\": float((pred == ref_h5.argmax(1)).mean()),\n           \"rows_differing_argmax_vs_blend_H5_dev\": np.where(pred != ref_h5.argmax(1))[0].tolist(),\n           \"max_abs_proba_diff_vs_blend_H5_dev\": float(np.abs(blend - ref_h5).max()),\n           \"max_abs_proba_diff_vs_shared_pass_reference\": float(np.abs(blend - ref_shared).max()),\n           \"components\": {k: {\"balanced_accuracy\": float(balanced_accuracy_score(y, comps[k].argmax(1))),\n                              \"max_abs_proba_diff_vs_reference\": float(np.abs(comps[k] - ref_comps[k]).max()),\n                              \"argmax_agreement_vs_reference\": float((comps[k].argmax(1) == ref_comps[k].argmax(1)).mean())}\n                          for k in H5_COMPONENTS}}\n    fails = []\n    if abs(bal - expected_bal_acc) > tol:\n        fails.append(f\"dev balanced accuracy {bal:.6f} differs from {expected_bal_acc:.6f} by more than {tol}\")\n    if rep[\"argmax_agreement_vs_blend_H5_dev\"] < 1.0:\n        fails.append(f\"argmax differs from blend_H5_dev.csv on rows {rep['rows_differing_argmax_vs_blend_H5_dev']}\")\n    if rep[\"max_abs_proba_diff_vs_shared_pass_reference\"] > max_dp_shared:\n        fails.append(f\"max |p - shared-pass reference| {rep['max_abs_proba_diff_vs_shared_pass_reference']:.3g} > {max_dp_shared}\")\n    return rep, fails\n\n\ndef validate_submission(sub, test, class_names):\n    assert list(sub.columns) == [\"id\", \"model\"], list(sub.columns)\n    assert len(sub) == len(test), (len(sub), len(test))\n    assert sub[\"id\"].notna().all() and sub[\"id\"].is_unique, \"ids must be unique and non-null\"\n    assert sub[\"id\"].tolist() == test[\"id\"].tolist(), \"ids must match test.csv (same order)\"\n    assert sub[\"model\"].notna().all() and set(sub[\"model\"]) <= set(class_names), \"labels outside the 18 classes\"\n    return True\n\n\ndef extrapolate(batch_seconds, batch_rows, n_rows=4000, batch_size=8):\n    \"\"\"Linear extrapolation of the shared forward pass from measured per-batch wall-clock.\"\"\"\n    secs, rows = np.asarray(batch_seconds, dtype=float), np.asarray(batch_rows, dtype=float)\n    per_row = secs.sum() / rows.sum()\n    return {\"measured_batches\": int(len(secs)), \"measured_rows\": int(rows.sum()), \"measured_seconds\": float(secs.sum()),\n            \"mean_seconds_per_batch\": float(secs.mean()), \"p95_seconds_per_batch\": float(np.percentile(secs, 95)),\n            \"seconds_per_row\": float(per_row), \"rows\": n_rows, \"batches\": int(np.ceil(n_rows / batch_size)),\n            \"estimated_forward_minutes\": float(per_row * n_rows / 60)}\n", "import joblib\nfrom sklearn.model_selection import GroupShuffleSplit\n\nchecks = {\"adapter/adapter_model.safetensors\": f\"{ADAPTER_DIR}/adapter_model.safetensors\",\n          \"adapter/adapter_config.json\": f\"{ADAPTER_DIR}/adapter_config.json\",\n          \"adapter/label_map.json\": f\"{ADAPTER_DIR}/label_map.json\",\n          **{k: f\"{H5_DIR}/{k}\" for k in [\"probe_armh.npz\", \"stylo_exp03_model.joblib\", \"blend_H5_dev.csv\", \"h5_dev_reference.csv\"]}}\nfor k, path in checks.items():\n    got = sha256_file(path)\n    assert got == SHA[k], f\"{k}: sha256 {got} != expected {SHA[k]}\"\nprint(\"all artifact sha256 verified:\", list(checks))\n\ntest = pd.read_csv(f\"{DATA_DIR}/test.csv\")  # row count not assumed: the hidden rerun has ~4,000 rows\nassert {\"id\", \"text\"} <= set(test.columns) and len(test) > 0\nprint(\"test:\", test.shape)\n\nlabel2id = {n: i for i, n in enumerate(CLASS_NAMES)}\nid2label = {i: n for n, i in label2id.items()}\nif RUN_DEV_CHECK:\n    TRAIN_CSV = f\"{DATA_DIR}/train.csv\"\n    if not os.path.exists(TRAIN_CSV):\n        print(f\"WARNING: run_dev_check=true but {TRAIN_CSV} does not exist; dev check skipped\")\n        RUN_DEV_CHECK = False\nif RUN_DEV_CHECK:\n    got = sha256_file(TRAIN_CSV)\n    assert got == SHA[\"train.csv\"], f\"train.csv: sha256 {got} != expected {SHA['train.csv']}\"\n    train = pd.read_csv(TRAIN_CSV)\n    df = train[[\"id\", \"prompt_id\", \"text\", \"model\"]].copy()\n    df[\"labels\"] = df[\"model\"].map(label2id).astype(int)\n    _, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(df, groups=df[\"prompt_id\"]))\n    dev_df = df.iloc[va_idx].reset_index(drop=True)\n    assert (len(dev_df), dev_df[\"prompt_id\"].nunique()) == (DC[\"rows\"], DC[\"prompts\"])\n    assert (dev_df[\"model\"].value_counts() == 23).all() and dev_df[\"model\"].nunique() == N_CLASSES\n    print(\"train.csv sha256 verified; dev split verified: 414 rows / 23 prompts (exp01 split)\")\nprint(\"run_dev_check:\", RUN_DEV_CHECK)\n\nwith open(f\"{ADAPTER_DIR}/label_map.json\") as f:\n    label_map_names = [n for _, n in sorted((int(k), v) for k, v in json.load(f)[\"id2label\"].items())]\nwith open(f\"{ADAPTER_DIR}/adapter_config.json\") as f:\n    adapter_cfg = json.load(f)\n# PEFT appends its SEQ_CLS defaults (\"classifier\", \"score\") to the trained modules_to_save=[\"score\"].\nassert \"score\" in adapter_cfg[\"modules_to_save\"], adapter_cfg[\"modules_to_save\"]\nassert set(adapter_cfg[\"modules_to_save\"]) <= {\"score\", \"classifier\"}, adapter_cfg[\"modules_to_save\"]\nassert adapter_cfg[\"task_type\"] == \"SEQ_CLS\", adapter_cfg[\"task_type\"]\nprobe = ProbeModel(f\"{H5_DIR}/probe_armh.npz\")\nassert probe.meta[\"C\"] == 0.03 and probe.meta[\"model\"] == \"probe_armh\", probe.meta.get(\"C\")\nstylo = joblib.load(f\"{H5_DIR}/stylo_exp03_model.joblib\")\nprint(\"probe C:\", probe.meta[\"C\"], \"| coef\", probe.coef.shape, \"| stylo:\", [s for s, _ in stylo.steps])\n", "from peft import PeftModel, prepare_model_for_kbit_training\nfrom transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig, set_seed\n\nt0 = time.time()\ntokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)\nqcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\"nf4\", bnb_4bit_compute_dtype=torch.float16,\n                          bnb_4bit_use_double_quant=True)\nset_seed(SEED)\nbase = AutoModelForSequenceClassification.from_pretrained(\n    MODEL_PATH, num_labels=N_CLASSES, id2label=id2label, label2id=label2id, quantization_config=qcfg,\n    device_map={\"\": 0}, torch_dtype=torch.float16, local_files_only=True)\nbase.config.pad_token_id = tokenizer.pad_token_id\nassert type(base).__name__ == \"Qwen3ForSequenceClassification\", type(base).__name__\n# Same dtype state as the Arm H kernel that produced the probe features and head logits: prepare_model_for_kbit_training\n# upcasts the non-quantized params (embeddings, RMSNorms, score) to fp32. Without it the dev check fails\n# (kernel v2: bal-acc 0.4300, max |dp| 0.017). No gradient checkpointing, all params frozen.\nbase = prepare_model_for_kbit_training(base, use_gradient_checkpointing=False)\nassert base.model.norm.weight.dtype == torch.float32 and base.score.weight.dtype == torch.float32\nmodel = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)\nassert not any(p.requires_grad for p in model.parameters())\nmodel.eval()\nscore_wrapper = model.base_model.model.score  # ModulesToSaveWrapper; the hook captures its input\nassert \"default\" in score_wrapper.modules_to_save\nload_minutes = (time.time() - t0) / 60\nprint(f\"model + adapter loaded in {load_minutes:.1f} min | GPU allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB\")\n\nmodel_config_names = [model.config.id2label[i] for i in range(N_CLASSES)]\ncheck_class_order(label_map_names, model_config_names, probe.class_names, stylo.classes_)\nif RUN_DEV_CHECK:\n    check_class_order(label_map_names, model_config_names, probe.class_names, stylo.classes_, train[\"model\"])\nprint(\"class order identical across CLASS_NAMES, head config, label_map.json, probe npz and stylo classes_:\", CLASS_NAMES)\n", "batch_log = []\n\n\ndef shared_forward(texts, split):\n    \"\"\"One forward pass per batch of 8 (file order): pooled_forward returns head logits + pooled hidden states.\"\"\"\n    hidden = np.zeros((len(texts), probe.mean.shape[0]), dtype=np.float32)\n    logits = np.zeros((len(texts), N_CLASSES), dtype=np.float32)\n    for start in range(0, len(texts), BATCH):\n        chunk = texts[start:start + BATCH]\n        t = time.perf_counter()\n        h, lg = pooled_forward(model, score_wrapper, tokenizer, chunk, BATCH, MAX_LENGTH, N_CLASSES)\n        torch.cuda.synchronize()\n        dt = time.perf_counter() - t\n        hidden[start:start + len(chunk)], logits[start:start + len(chunk)] = h, lg\n        batch_log.append({\"split\": split, \"start\": start, \"rows\": len(chunk), \"seconds\": dt,\n                          \"peak_allocated_gb\": torch.cuda.max_memory_allocated() / 1e9})\n        if (start // BATCH) % 25 == 0:\n            print(f\"[{split}] batch {start // BATCH + 1}/{-(-len(texts) // BATCH)}: {dt:.2f}s\")\n    return hidden, logits\n\n\ndef predict_h5(texts, split):\n    hidden, logits = shared_forward(list(texts), split)\n    t = time.perf_counter()\n    comps = component_probas(hidden, logits, texts, probe, stylo)\n    blend = h5_blend(comps)\n    print(f\"[{split}] probe + stylo + blend on CPU: {time.perf_counter() - t:.1f}s\")\n    return comps, blend, logits\n\n\ntorch.cuda.reset_peak_memory_stats()\n", "# Dev reproduction (414 rows, only when run_dev_check=true): same inference path, compared with the reported H5\n# before any test prediction. Saved version 3 passed it end-to-end (bal-acc 0.432367, 414/414 argmax, max |dp| 5e-11).\nif RUN_DEV_CHECK:\n    t0 = time.time()\n    dev_comps, dev_blend, dev_logits = predict_h5(dev_df[\"text\"].tolist(), \"dev\")\n    dev_minutes = (time.time() - t0) / 60\n\n    def aligned(path):\n        r = pd.read_csv(path).set_index(\"id\").loc[dev_df[\"id\"]].reset_index()\n        assert (r[\"labels\"].to_numpy() == dev_df[\"labels\"].to_numpy()).all()\n        return r\n\n    PCOLS = [f\"p_{n}\" for n in CLASS_NAMES]\n    h5_rep = aligned(f\"{H5_DIR}/blend_H5_dev.csv\")[PCOLS].to_numpy(np.float64)\n    h5_rep = h5_rep / h5_rep.sum(1, keepdims=True)\n    ref = aligned(f\"{H5_DIR}/h5_dev_reference.csv\")\n    ref_shared = ref[[f\"h5:{c}\" for c in PCOLS]].to_numpy(np.float64)\n    ref_comps = {k: ref[[f\"{k}:{c}\" for c in PCOLS]].to_numpy(np.float64) for k in H5_COMPONENTS}\n    dev_report, dev_failures = dev_reproduction(\n        dev_df[\"labels\"].to_numpy(), dev_blend, dev_comps, h5_rep, ref_shared, ref_comps,\n        DC[\"expected_balanced_accuracy\"], DC[\"tolerance\"], DC[\"max_abs_proba_diff_vs_shared_pass_reference\"])\n    dev_report[\"minutes\"] = dev_minutes\n    print(json.dumps(dev_report, indent=1))\n    out = dev_df[[\"id\", \"prompt_id\", \"labels\"]].copy()\n    out = pd.concat([out, pd.DataFrame(dev_blend, columns=PCOLS)], axis=1)\n    out.to_csv(f\"{OUT}/h5_dev_predictions.csv\", index=False, float_format=\"%.10g\")\n    with open(f\"{OUT}/h5_dev_check.json\", \"w\") as f:\n        json.dump(dev_report, f, indent=1)\n    if dev_failures:\n        raise RuntimeError(\"dev reproduction failed; investigate before writing a submission: \" + \"; \".join(dev_failures))\n    print(f\"dev reproduction OK: balanced accuracy {dev_report['balanced_accuracy']:.4f} \"\n          f\"(expected {DC['expected_balanced_accuracy']:.4f} +/- {DC['tolerance']}), argmax identical to blend_H5_dev.csv\")\nelse:\n    dev_report = {\"skipped\": True}\n    dev_minutes = 0\n    print(\"dev check skipped (run_dev_check=false)\")\n", "import pyarrow as pa\nimport pyarrow.parquet as pq\n\ntext_ok = test[\"text\"].map(lambda v: isinstance(v, str)).astype(bool)\nprint(f\"test.csv: {int((~text_ok).sum())} null/non-string text rows coerced to empty string\")\ntest[\"text\"] = test[\"text\"].where(text_ok, \"\").fillna(\"\")\n\nt0 = time.time()\ntest_comps, test_blend, _ = predict_h5(test[\"text\"].tolist(), \"test\")\ntest_minutes = (time.time() - t0) / 60\nsubmission = pd.DataFrame({\"id\": test[\"id\"].to_numpy(), \"model\": [CLASS_NAMES[i] for i in test_blend.argmax(1)]})\nvalidate_submission(submission, test, CLASS_NAMES)\nsample_schema = pq.read_schema(f\"{DATA_DIR}/sample_submission.parquet\")\nassert sample_schema.names == [\"id\", \"model\"], sample_schema\npq.write_table(pa.Table.from_pandas(submission, preserve_index=False).cast(sample_schema), f\"{OUT}/submission.parquet\")\n\nwritten = pd.read_parquet(f\"{OUT}/submission.parquet\")\nvalidate_submission(written, test, CLASS_NAMES)\nassert pq.read_schema(f\"{OUT}/submission.parquet\").equals(sample_schema, check_metadata=False)\nprint(f\"submission.parquet written and validated: {len(written)} rows, columns {list(written.columns)}, {test_minutes:.2f} min\")\n\n# Diagnostics only from here on: submission.parquet is final, so nothing below may raise.\ntry:\n    print(written.head(10))\n    print(written[\"model\"].value_counts().to_string())\nexcept Exception as e:\n    print(f\"WARNING: post-submission diagnostics failed: {type(e).__name__}: {e}\")\n", "# Post-submission diagnostics (batch log, runtime extrapolation, run summary): warnings only, never raises.\nimport urllib.request\n\ntry:\n    try:\n        urllib.request.urlopen(\"https://pypi.org\", timeout=5)\n        internet = \"REACHABLE\"\n    except Exception as e:\n        internet = f\"unreachable ({type(e).__name__})\"\n    print(\"internet:\", internet)\n\n    bl = pd.DataFrame(batch_log)\n    bl.to_csv(f\"{OUT}/h5_batch_log.csv\", index=False)\n    peak = {\"allocated_gb\": torch.cuda.max_memory_allocated() / 1e9, \"reserved_gb\": torch.cuda.max_memory_reserved() / 1e9}\n    est = extrapolate(bl[\"seconds\"], bl[\"rows\"], n_rows=4000, batch_size=BATCH)\n    total_minutes = (time.time() - RUN_START) / 60\n    est[\"estimated_total_minutes_4000_rows\"] = (total_minutes - test_minutes) + est[\"estimated_forward_minutes\"]\n    summary = {\"status\": \"completed\", \"test_rows\": len(test), \"run_dev_check\": RUN_DEV_CHECK,\n               \"dev_balanced_accuracy\": dev_report.get(\"balanced_accuracy\"),\n               \"dev_check\": dev_report, \"peak_gpu_memory\": peak, \"model_load_minutes\": load_minutes,\n               \"dev_minutes\": dev_minutes, \"test_minutes\": test_minutes, \"total_minutes\": total_minutes,\n               \"per_batch_seconds\": {s: g[\"seconds\"].describe().to_dict() for s, g in bl.groupby(\"split\")},\n               \"extrapolation\": est, \"internet\": internet, \"versions\": versions, \"config\": CONFIG}\n    with open(f\"{OUT}/h5_run_summary.json\", \"w\") as f:\n        json.dump(summary, f, indent=1, default=float)\n    print(json.dumps({k: v for k, v in summary.items() if k not in (\"config\", \"dev_check\")}, indent=1, default=float))\nexcept Exception as e:\n    print(f\"WARNING: run summary failed (submission.parquet already written): {type(e).__name__}: {e}\")\n"]""")
sim_ns = {"__name__": "__main__"}
sim_error, sim_failed_cell = None, None
for i, src in enumerate(CELLS):
    try:
        exec(compile(src, f"h5-cell-{i + 2}", "exec"), sim_ns)
    except Exception:
        sim_error, sim_failed_cell = traceback.format_exc(), i + 2
        print(sim_error)
        break

import pandas as pd
import torch
summary_path = "/kaggle/working/h5_run_summary.json"
try:
    summary = json.load(open(summary_path)) if sim_error is None else {}
except Exception:
    summary = {}
bl = pd.DataFrame(sim_ns.get("batch_log", []))
if len(bl) and "tokenizer" in sim_ns and "test" in sim_ns:
    tok = sim_ns["tokenizer"]
    texts = sim_ns["test"]["text"].tolist()
    lens = [len(x) for x in tok(texts, truncation=True, max_length=sim_ns["MAX_LENGTH"])["input_ids"]]
    tb = bl[bl["split"] == "test"].copy()
    tb["max_tokens"] = [max(lens[s:s + r]) for s, r in zip(tb["start"], tb["rows"])]
    tb["full_maxlen_batch"] = [min(lens[s:s + r]) == sim_ns["MAX_LENGTH"] for s, r in zip(tb["start"], tb["rows"])]
    tb.to_csv("/kaggle/working/h5_batch_log.csv", index=False)
    summary["sim_batches"] = {
        "test_batches": int(len(tb)), "truncated_rows": int(sum(l == sim_ns["MAX_LENGTH"] for l in lens)),
        "seconds_all": tb["seconds"].describe().to_dict(),
        "seconds_full_512_token_batches": tb.loc[tb["full_maxlen_batch"], "seconds"].describe().to_dict(),
        "seconds_by_max_tokens_bucket": {str(k): v for k, v in tb.groupby(
            pd.cut(tb["max_tokens"], [0, 128, 256, 384, 511, 512]), observed=True)["seconds"]
            .describe()[["count", "mean", "max"]].to_dict("index").items()}}
summary.update({"simulation": SIM_INFO, "exception": sim_error, "failed_cell": sim_failed_cell,
                "status": summary.get("status", "exception" if sim_error else "incomplete"),
                "peak_gpu_memory": summary.get("peak_gpu_memory") or (
                    {"allocated_gb": torch.cuda.max_memory_allocated() / 1e9,
                     "reserved_gb": torch.cuda.max_memory_reserved() / 1e9} if torch.cuda.is_available() else None),
                "submission_parquet_written": __import__("os").path.exists("/kaggle/working/submission.parquet")})
text = json.dumps(summary, indent=1, default=str)  # serialise fully before touching the file
with open(summary_path, "w") as f:
    f.write(text)
print(json.dumps({k: v for k, v in summary.items() if k not in ("config", "versions", "dev_check")}, indent=1, default=str))
